# 🛡️ Z-Guard AI: Real-Time Server Anomaly Detection
### IBM Z Datathon 2026 Interactive Notebook

This notebook provides a complete, interactive walkthrough of the **Z-Guard AI** pipeline:
1. **Environment & Hardware Inspection** (CPU Architecture & s390x detection)
2. **Data Ingestion & Cleaning** (Synthetic Diurnal Telemetry + Injected Incidents)
3. **Unsupervised Machine Learning** (Scikit-Learn Isolation Forest)
4. **Metric Attribution & Root Cause Explainability**
5. **Interactive Visualizations** (Plotly Timelines, Radar Charts, Scatter Plots)
6. **Live Hardware Telemetry Capture** (via `psutil`)

## Step 1: Install & Import Dependencies

In [ ]:
# Install requirements if running in a fresh Jupyter / Colab environment
# !pip install -r ../requirements.txt

import os
import sys
import numpy as np
import pandas as pd

# Add project root to sys.path so we can import src modules
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import FEATURE_COLUMNS, METRIC_LABELS, METRIC_UNITS, OPERATIONAL_THRESHOLDS
from src.data_loader import generate_synthetic_telemetry, validate_and_clean_data
from src.detector import AnomalyDetector
from src.metrics_collector import SystemMetricsCollector
from src.visualizer import (
    create_time_series_chart,
    create_synchronized_metrics_dashboard,
    create_scatter_correlation,
    create_radar_attribution
)

print("✓ Z-Guard AI modules successfully imported!")

## Step 2: Check Hardware & Host Architecture (s390x / IBM LinuxONE Readiness)

In [ ]:
collector = SystemMetricsCollector()
host_info = collector.get_system_info()

print(f"OS Platform:       {host_info.get('os')} ({host_info.get('machine_arch')})")
print(f"CPU Cores:         {host_info.get('cpu_cores_logical')}")
print(f"System Memory:     {host_info.get('memory_total_gb')}")
print(f"IBM s390x Target:  {'YES (Mainframe hardware detected)' if host_info.get('is_s390x') else 'NO (Running in development mode)'}")

## Step 3: Load & Validate Server Telemetry
You can load the pre-packaged demo dataset (`sample_server_metrics.csv`) or upload your own CSV.

In [ ]:
csv_path = os.path.join(project_root, "data", "sample_server_metrics.csv")
raw_df = pd.read_csv(csv_path)

# Validate and clean dataset
df, report = validate_and_clean_data(raw_df)
print(f"Validation Status: {report['is_valid']}")
print(f"Total Records:     {len(df)}")
df.head()

## Step 4: Train Scikit-Learn Isolation Forest Model
Fit the unsupervised anomaly detector using `RobustScaler` and configurable `contamination`.

In [ ]:
# Initialize and fit model
detector = AnomalyDetector(contamination=0.05, n_estimators=150, random_state=42)
detector.fit(df)

print("Model summary:")
detector.get_model_summary()

## Step 5: Run Inference & Root-Cause Metric Attribution

In [ ]:
scored_df = detector.predict(df)

anomalies = scored_df[scored_df["anomaly"] == True]
print(f"Total Records Analyzed: {len(scored_df)}")
print(f"Detected Anomalies:     {len(anomalies)} ({len(anomalies)/len(scored_df)*100:.1f}%)")
print(f"Critical Severities:    {(scored_df['severity'] == 'CRITICAL').sum()}")
print(f"Warning Severities:     {(scored_df['severity'] == 'WARNING').sum()}")

# Display top detected anomaly events
anomalies[["timestamp", "severity", "anomaly_score", "culprit_metric", "anomaly_reason", "mitigation_action"]].head(10)

## Step 6: Interactive Plotly Visualizations
Plotly charts display interactively right inside your Jupyter Notebook.

In [ ]:
# 1. Synchronized 4-metric timeline with highlighted anomalies
fig_sync = create_synchronized_metrics_dashboard(scored_df)
fig_sync.show()

In [ ]:
# 2. Single metric deep-dive (e.g. CPU or Memory with threshold lines)
fig_cpu = create_time_series_chart(scored_df, "cpu_pct")
fig_cpu.show()

In [ ]:
# 3. Multi-metric Incident Root-Cause Radar Deep-Dive
if not anomalies.empty:
    sample_anomaly_row = anomalies.iloc[0]
    print(f"Radar analysis for anomaly at {sample_anomaly_row['timestamp']} (Primary driver: {sample_anomaly_row['culprit_metric']})")
    fig_radar = create_radar_attribution(sample_anomaly_row, detector.baseline_stats)
    fig_radar.show()

## Step 7: Live Hardware Snapshot via psutil

In [ ]:
snapshot = collector.capture_snapshot()
live_point = pd.DataFrame([snapshot])
live_score = detector.predict(live_point)

print("Live Snapshot from host machine:")
print(f"Timestamp:   {snapshot['timestamp']}")
print(f"CPU:         {snapshot['cpu_pct']}%")
print(f"Memory:      {snapshot['memory_pct']}%")
print(f"Disk:        {snapshot['disk_pct']}%")
print(f"Network:     {snapshot['network_mbps']} Mbps")
print(f"ML Verdict:  {live_score.loc[0, 'prediction_label']} (Severity Score: {live_score.loc[0, 'anomaly_score']}%)")

## Step 8: How to Run the Full Streamlit Dashboard
To launch the full interactive web dashboard from Jupyter, execute the following cell:

In [ ]:
# Run this in your terminal or use the line magic below:
# !streamlit run ../app.py